In [1]:
!pip uninstall plotly_cloud -y

Found existing installation: plotly-cloud 0.3.0
Uninstalling plotly-cloud-0.3.0:
  Successfully uninstalled plotly-cloud-0.3.0


In [ ]:
#pip install dash plotly

In [18]:
import os
from threading import Timer
import webbrowser

from dash import Dash, dcc, html, Input, Output, State
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA = {
    "hero_stats": [
        ("55,500", "Patient Records"),
        ("7",      "Dual-Task Models"),
        ("79.9%",  "Best LoS R2"),
        ("62%",    "Best Diag Acc"),
        ("22",     "Bottleneck Hospitals"),
        ("8.39d",  "LoS Gap"),
    ],
    "models": [
        {"rank":1,"short":"GBT-GBT","pair":"GBT Regressor → GBT Classifier","family":"Boosted Trees","stars":5,"r2":0.7934,"rmse":3.94,"mae":1.80,"acc":62,"f1":0.61,"summary":"Best combined LoS + diagnosis performance.","delay":"Non-linear compounding: Cancer + Emergency at slow hospitals reaches 15.99d worst case.","diag":"Predicted LoS > 14d maps to Cancer/Heart Disease at 67% probability."},
        {"rank":2,"short":"LR-LogReg","pair":"Linear Regression → Logistic Regression","family":"Statistical","stars":5,"r2":0.7988,"rmse":3.89,"mae":1.80,"acc":60,"f1":0.59,"summary":"Best R2 and fastest inference.","delay":"Hospital average coefficient dominates the formula and explains most delay variance.","diag":"Multinomial Logistic Regression maps predicted LoS into 6 diagnosis classes."},
        {"rank":3,"short":"DT-DT","pair":"Decision Tree Regressor → Decision Tree Classifier","family":"Tree-Based","stars":4,"r2":0.7980,"rmse":3.90,"mae":1.83,"acc":55,"f1":0.54,"summary":"Most explainable; hospital importance 99.98%.","delay":"Every split starts with hospital baseline, showing the hospital itself is the bottleneck.","diag":"Rules are printable and fully auditable for review boards."},
        {"rank":4,"short":"GBTCV-RF","pair":"GBT CrossValidation → Random Forest Classifier","family":"Tuned Ensembles","stars":4,"r2":0.7975,"rmse":3.90,"mae":1.90,"acc":60,"f1":0.59,"summary":"72 distributed validation runs.","delay":"Cross-validation confirms bottleneck patterns are stable and not a lucky split.","diag":"Validated Stage 1 LoS feeds a stable RF diagnosis model."},
        {"rank":5,"short":"RF-RF","pair":"Random Forest Regressor → Random Forest Classifier","family":"Full Ensembles","stars":3,"r2":0.7356,"rmse":4.46,"mae":3.15,"acc":55,"f1":0.54,"summary":"Stable but weaker than GBT.","delay":"Averaging smooths the dominant signal, reducing sharp bottleneck detection.","diag":"Stage 1 noise lowers Stage 2 diagnosis quality."},
        {"rank":6,"short":"GLM-LR","pair":"GLM Poisson → Logistic Regression","family":"Statistical GLM","stars":3,"r2":0.7470,"rmse":4.36,"mae":3.10,"acc":57,"f1":0.56,"summary":"Most statistically principled for count data.","delay":"Poisson models multiplicative delay effects like Emergency x2.1 vs Elective.","diag":"Confidence intervals make it attractive for formal reporting."},
        {"rank":7,"short":"FM-FM","pair":"Factorization Machine Regressor → FM Classifier","family":"Fact. Machines","stars":2,"r2":-1.5668,"rmse":13.89,"mae":11.50,"acc":58,"f1":0.57,"summary":"Research model with useful interaction discovery.","delay":"Regression failed because the step size was too small, but interactions were still informative.","diag":"Classifier uses factor vectors directly and still performs reasonably."},
    ],
}

GRAPH_PALETTES = {
    "Neon Blaze":    ["#ff2d78","#00e5ff","#ffe600","#ff6d00","#b400ff"],
    "Electric Ocean":["#00cfff","#ff4ecd","#39ff14","#ff9500","#7b2fff"],
    "Vivid Carnival":["#ff3cac","#784ba0","#2b86c5","#ff9a3c","#29ffc6"],
}

DARK  = {"bg":"#1e1b2e","surface":"#13111f","surface2":"#2a2640","border":"#4c4878","text":"#f0ecff","muted":"#a89ec9","accent":"#ff2d78"}
LIGHT = {"bg":"#fff0f3","surface":"#ffe4ec","surface2":"#ffd6e3","border":"#ffadc9","text":"#3d1a2b","muted":"#9e5e75","accent":"#e0006e"}

def theme(mode): return DARK if mode == "dark" else LIGHT
def gpal(name):  return GRAPH_PALETTES.get(name, list(GRAPH_PALETTES.values())[0])

def stat_card(v, l, mode):
    c = theme(mode)
    return html.Div(
        [html.Div(v, style={"fontSize":"1.4rem","fontWeight":700,"color":c["accent"]}),
         html.Div(l, style={"fontSize":"0.75rem","color":c["muted"],"marginTop":"0.15rem"})],
        style={"background":c["surface2"],"border":f"1px solid {c['border']}",
               "borderRadius":"12px","padding":"0.8rem 1.1rem","minWidth":"120px"})

def star_rating(n):
    return html.Span(
        [html.Span("★", style={"color":"#ff2d78"}) for _ in range(n)] +
        [html.Span("★", style={"color":"#4c4878"}) for _ in range(5 - n)])

def hex_rgba(h, a=0.25):
    h = h.lstrip("#")
    r, g, b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    return f"rgba({r},{g},{b},{a})"

def build_figure(chart_group, chart_variant, gpal_name, mode):
    c = theme(mode); p = gpal(gpal_name)
    ms   = [m["short"] for m in DATA["models"]]
    r2   = [max(0, m["r2"])  for m in DATA["models"]]
    rmse = [m["rmse"] for m in DATA["models"]]
    mae  = [m["mae"]  for m in DATA["models"]]
    acc  = [m["acc"]  for m in DATA["models"]]
    gc = "rgba(200,180,230,0.25)"
    base = dict(
        paper_bgcolor=c["surface"], plot_bgcolor=c["surface2"],
        font=dict(color=c["text"], family="Space Grotesk,sans-serif", size=13),
        margin=dict(l=55,r=45,t=48,b=55),
        legend=dict(orientation="h",yanchor="bottom",y=1.02,xanchor="right",x=1,bgcolor="rgba(0,0,0,0)"),
        xaxis=dict(gridcolor=gc,color=c["text"],showline=True,linecolor=c["border"]),
        yaxis=dict(gridcolor=gc,color=c["text"],showline=True,linecolor=c["border"]))

    if chart_group == "r2comp":
        if chart_variant == "scatter":
            fig = go.Figure()
            for i, m in enumerate(DATA["models"]):
                fig.add_trace(go.Scatter(
                    x=[max(0,m["r2"])], y=[m["acc"]], mode="markers+text",
                    name=m["short"], text=[m["short"]], textposition="top center",
                    marker=dict(size=m["f1"]*80, color=p[i%len(p)],
                                line=dict(width=2,color="#fff"), opacity=0.9)))
            fig.update_layout(title="LoS R2 vs Diagnosis Accuracy (bubble = F1)",
                              xaxis_title="LoS R2", yaxis_title="Diagnosis Acc (%)", **base)
            return fig
        if chart_variant == "radar":
            cats = ["LoS R2","Diag Acc","F1","Low RMSE","Low MAE"]
            mr = max(rmse); mm = max(mae)
            fig = go.Figure()
            for i, m in enumerate(DATA["models"]):
                v = [max(0,m["r2"]),m["acc"]/100,m["f1"],1-m["rmse"]/mr,1-m["mae"]/mm]; v += [v[0]]
                fig.add_trace(go.Scatterpolar(r=v, theta=cats+[cats[0]], name=m["short"],
                    line=dict(color=p[i%len(p)],width=2), fill="toself", opacity=0.35))
            fig.update_layout(title="Model Radar: R2 Acc F1 Error",
                polar=dict(bgcolor=c["surface2"],radialaxis=dict(visible=True,color=c["muted"])),
                paper_bgcolor=c["surface"],
                font=dict(color=c["text"],family="Space Grotesk,sans-serif"),
                margin=dict(l=55,r=45,t=48,b=55),
                legend=dict(orientation="h",yanchor="bottom",y=1.02,xanchor="right",x=1,bgcolor="rgba(0,0,0,0)"))
            return fig
        fig = go.Figure()
        fig.add_trace(go.Bar(name="LoS R2", x=ms, y=r2, marker_color=p[0], opacity=0.9))
        fig.add_trace(go.Bar(name="Diag Acc /100", x=ms, y=[a/100 for a in acc], marker_color=p[1], opacity=0.9))
        fig.update_layout(title="LoS R2 and Diagnosis Accuracy per Model",
                          barmode="group", xaxis_title="Model", yaxis_title="Score", **base)
        return fig

    if chart_group == "errcomp":
        if chart_variant == "grouped_bar":
            fig = go.Figure()
            fig.add_trace(go.Bar(name="RMSE", x=ms, y=rmse, marker_color=p[2], opacity=0.9))
            fig.add_trace(go.Bar(name="MAE",  x=ms, y=mae,  marker_color=p[3%len(p)], opacity=0.9))
            fig.update_layout(title="RMSE vs MAE per Model", barmode="group",
                              xaxis_title="Model", yaxis_title="Days", **base)
            return fig
        if chart_variant == "scatter":
            fig = go.Figure()
            for i, m in enumerate(DATA["models"]):
                fig.add_trace(go.Scatter(x=[m["rmse"]], y=[m["mae"]], mode="markers+text",
                    text=[m["short"]], textposition="top center", name=m["short"],
                    marker=dict(size=18, color=p[i%len(p)], line=dict(width=2,color="#fff"), opacity=0.9)))
            fig.update_layout(title="RMSE vs MAE Scatter",
                              xaxis_title="RMSE (days)", yaxis_title="MAE (days)", **base)
            return fig
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=ms, y=rmse, name="RMSE", mode="lines+markers",
            line=dict(color=p[0],width=3), marker=dict(size=10,color=p[0]),
            fill="tozeroy", fillcolor=hex_rgba(p[0])))
        fig.add_trace(go.Scatter(x=ms, y=mae, name="MAE", mode="lines+markers",
            line=dict(color=p[1],width=3), marker=dict(size=10,color=p[1]),
            fill="tozeroy", fillcolor=hex_rgba(p[1])))
        fig.update_layout(title="RMSE and MAE Area Comparison",
                          xaxis_title="Model", yaxis_title="Days", **base)
        return fig

    labels = ["Rows Processed","LoS Computed","Bottlenecks Flagged",
              "SQL Queries","Hospitals Clustered","CrossVal Runs","Outputs Saved"]
    before = [5000,0,0,0,0,0,0]
    after  = [55500,55500,55500,15,22,72,7]

    if chart_variant == "lollipop":
        fig = go.Figure()
        for i, (lb, bf, af) in enumerate(zip(labels, before, after)):
            fig.add_trace(go.Scatter(x=[bf,af], y=[lb,lb], mode="lines",
                line=dict(color=p[i%len(p)],width=3), showlegend=False))
            fig.add_trace(go.Scatter(x=[bf], y=[lb], mode="markers",
                marker=dict(color=p[(i+2)%len(p)],size=12,line=dict(width=2,color="#fff")),
                name="Before", showlegend=(i==0)))
            fig.add_trace(go.Scatter(x=[af], y=[lb], mode="markers",
                marker=dict(color=p[i%len(p)],size=16,symbol="star",line=dict(width=2,color="#fff")),
                name="After", showlegend=(i==0)))
        bl = {**base, "xaxis":{**base["xaxis"],"type":"log"}}
        fig.update_layout(title="Before vs After Spark Lollipop", **bl)
        return fig

    if chart_variant == "waterfall":
        delta = [a-b for a,b in zip(after,before)]
        fig = go.Figure(go.Waterfall(
            name="Uplift", orientation="v", x=labels, y=delta,
            connector=dict(line=dict(color=p[1],width=1,dash="dot")),
            increasing=dict(marker=dict(color=p[0])),
            decreasing=dict(marker=dict(color=p[3%len(p)])),
            totals=dict(marker=dict(color=p[2]))))
        fig.update_layout(title="Spark Uplift Waterfall",
                          xaxis_title="Metric", yaxis_title="Delta Value", **base)
        return fig

    bl = {**base, "xaxis":{**base["xaxis"],"type":"log"}}
    fig = go.Figure()
    fig.add_trace(go.Bar(name="Before Spark", y=labels, x=before,
                         orientation="h", marker_color=p[3%len(p)], opacity=0.85))
    fig.add_trace(go.Bar(name="After Spark",  y=labels, x=after,
                         orientation="h", marker_color=p[0], opacity=0.85))
    fig.update_layout(title="Before vs After Spark Horizontal Bar",
                      barmode="group", xaxis_title="Value", yaxis_title="Metric", **bl)
    return fig

def build_report_charts(mode):
    c = theme(mode); p = list(GRAPH_PALETTES.values())[0]
    ms   = [m["short"] for m in DATA["models"]]
    acc  = [m["acc"]   for m in DATA["models"]]
    rmse = [m["rmse"]  for m in DATA["models"]]
    r2   = [max(0,m["r2"]) for m in DATA["models"]]
    f1   = [m["f1"]    for m in DATA["models"]]
    gc = "rgba(200,180,230,0.25)"
    base = dict(paper_bgcolor=c["surface"], plot_bgcolor=c["surface2"],
                font=dict(color=c["text"],family="Space Grotesk,sans-serif",size=12),
                margin=dict(l=50,r=30,t=44,b=50),
                xaxis=dict(gridcolor=gc,color=c["text"]),
                yaxis=dict(gridcolor=gc,color=c["text"]))
    figA = go.Figure(go.Pie(labels=ms, values=acc,
        marker=dict(colors=p[:len(ms)], line=dict(color="#fff",width=2)),
        textinfo="label+percent", hoverinfo="label+value", hole=0.35))
    figA.update_layout(title="Diagnosis Accuracy Share",
        paper_bgcolor=c["surface"],
        font=dict(color=c["text"],family="Space Grotesk,sans-serif",size=12),
        margin=dict(l=20,r=20,t=44,b=20), legend=dict(bgcolor="rgba(0,0,0,0)"))
    idx = sorted(range(len(ms)), key=lambda i: rmse[i])
    ms_s = [ms[i] for i in idx]; rmse_s = [rmse[i] for i in idx]
    col_s = [p[i%len(p)] for i in range(len(ms_s))]
    figB = go.Figure(go.Bar(y=ms_s, x=rmse_s, orientation="h",
        marker=dict(color=col_s,line=dict(width=1,color="#fff")),
        text=[f"{v:.2f}d" for v in rmse_s], textposition="outside"))
    figB.update_layout(title="RMSE Ranking best to worst", xaxis_title="RMSE (days)", **base)
    figC = go.Figure()
    for i, m in enumerate(DATA["models"]):
        figC.add_trace(go.Scatter(x=[max(0,m["r2"])], y=[m["f1"]], mode="markers+text",
            text=[m["short"]], textposition="top center", name=m["short"],
            marker=dict(size=20, color=p[i%len(p)], line=dict(width=2,color="#fff"), opacity=0.9)))
    figC.update_layout(title="LoS R2 vs Diagnosis F1",
                       xaxis_title="LoS R2", yaxis_title="Diag F1", **base)
    return figA, figB, figC

# ── INLINE CSS with DARK PINK + PURPLE DROPDOWNS, BLACK BOLD TEXT ──
INLINE_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@400;500;600;700&display=swap');
* { box-sizing: border-box; }
body, .app-shell { margin:0; font-family:'Space Grotesk',sans-serif; background:#1e1b2e; color:#f0ecff; }
.logo-text { font-size:1.25rem; font-weight:700; color:#ff2d78; letter-spacing:.02em; }
.logo-sub  { font-size:.72rem; color:#a89ec9; margin-top:.1rem; }
.custom-tabs-container { background:#13111f !important; border-bottom:1px solid #4c4878; }
.custom-tab { background:#13111f !important; color:#a89ec9 !important; border:none !important;
  border-bottom:3px solid transparent !important; padding:.65rem 1.4rem !important;
  font-family:'Space Grotesk',sans-serif; font-size:.88rem; font-weight:600; cursor:pointer; }
.custom-tab:hover { color:#ff2d78 !important; }
.custom-tab--selected { background:#1e1b2e !important; color:#ff2d78 !important; border-bottom:3px solid #ff2d78 !important; }
table { width:100%; border-collapse:collapse; font-size:.84rem; }
thead tr { background:#2a2640; }
th { padding:.6rem .7rem; text-align:left; color:#ff2d78; font-weight:700; white-space:nowrap; border-bottom:1px solid #4c4878; }
td { padding:.55rem .7rem; border-bottom:1px solid rgba(76,72,120,.4); vertical-align:top; }
tbody tr:hover { background:rgba(255,45,120,.06); }
.stat-v { font-size:1.35rem; font-weight:700; color:#ff2d78; }
.stat-l { font-size:.72rem; color:#a89ec9; margin-top:.1rem; }
::-webkit-scrollbar { width:6px; height:6px; }
::-webkit-scrollbar-track { background:#13111f; }
::-webkit-scrollbar-thumb { background:#4c4878; border-radius:99px; }

/* DROPDOWN: dark pink + purple gradient, black bold 14px text */
.Select-control, .dark-dd .Select-control {
  background: linear-gradient(135deg,#ff2d78,#784ba0) !important;
  border: 1px solid #4c4878 !important;
  color: #000000 !important;
  border-radius: 8px !important;
  font-weight: 700 !important;
  font-size: 14px !important;
}
.Select-menu-outer, .dark-dd .Select-menu-outer {
  background: linear-gradient(135deg,#ff99c9,#b388ff) !important;
  border: 1px solid #4c4878 !important;
  border-top:none !important;
  border-radius:0 0 8px 8px !important;
  z-index:9999 !important;
}
.Select-menu, .dark-dd .Select-menu {
  background: transparent !important;
  max-height:300px !important;
}
.Select-option, .dark-dd .Select-option {
  background-color: transparent !important;
  color:#000000 !important;
  font-weight:700 !important;
  font-size:14px !important;
  padding:.5rem .8rem !important;
  cursor:pointer !important;
}
.Select-option.is-focused, .dark-dd .Select-option.is-focused {
  background-color: rgba(255,255,255,0.4) !important;
  color:#000000 !important;
}
.Select-option.is-selected, .dark-dd .Select-option.is-selected {
  background-color: rgba(0,0,0,0.2) !important;
  color:#000000 !important;
  font-weight:700 !important;
}
.Select-value-label, .dark-dd .Select-value-label,
.Select-placeholder, .dark-dd .Select-placeholder {
  color:#000000 !important;
  font-weight:700 !important;
  font-size:14px !important;
}
.Select-arrow, .dark-dd .Select-arrow {
  border-top-color:#000000 !important;
}
.is-open > .Select-control, .dark-dd.is-open > .Select-control {
  background: linear-gradient(135deg,#ff2d78,#784ba0) !important;
  border-color:#ff2d78 !important;
}
.Select-input > input, .dark-dd .Select-input > input {
  color:#000000 !important;
  font-weight:700 !important;
  font-size:14px !important;
  background:transparent !important;
}
.VirtualizedSelectFocusedOption  { background-color:rgba(255,255,255,0.4) !important; color:#000000 !important; font-weight:700 !important; }
.VirtualizedSelectSelectedOption { background-color:rgba(0,0,0,0.2) !important;   color:#000000 !important; font-weight:700 !important; }
.VirtualizedSelectOption         { background-color:transparent !important;         color:#000000 !important; font-weight:700 !important; }
"""

app = Dash(__name__)
app.title = "BottleneckMiner"

# NEW IMAGE URL
LOGO_URL   = "https://tse4.mm.bing.net/th/id/OIP.nBmdN73Wg3KbOohlRWezzAHaEK?rs=1&pid=ImgDetMain&o=7&rm=3and"
KAGGLE_URL = "https://www.kaggle.com/datasets/prasad22/healthcare-dataset"

app.index_string = f"""<!DOCTYPE html>
<html>
<head>
{{%metas%}}
<title>{{%title%}}</title>
{{%favicon%}}
{{%css%}}
<style>{INLINE_CSS}</style>
</head>
<body>
{{%app_entry%}}
<footer>{{%config%}}{{%scripts%}}{{%renderer%}}</footer>
</body>
</html>"""

app.layout = html.Div([
    dcc.Store(id="theme-store", data="dark"),
    dcc.Store(id="gpal-store",  data=list(GRAPH_PALETTES.keys())[0]),

    html.Header([
        html.Div([
            html.Img(src=LOGO_URL, style={"height":"52px","borderRadius":"10px",
                     "marginRight":"0.8rem","objectFit":"cover"}),
            html.Div([html.Div("BottleneckMiner", className="logo-text"),
                      html.Div("Multi-Hospital Patient Journey", className="logo-sub")]),
        ], style={"display":"flex","alignItems":"center"}),
        html.Div([
            html.A("Kaggle Dataset", href=KAGGLE_URL, target="_blank",
                   style={"color":"#ff2d78","fontSize":"0.82rem","fontWeight":600,
                          "textDecoration":"none","marginRight":"1.2rem",
                          "border":"1px solid #ff2d78","borderRadius":"999px",
                          "padding":"0.3rem 0.8rem"}),
            html.Button("☀", id="theme-btn", n_clicks=0,
                        style={"background":"linear-gradient(135deg,#ff2d78,#ff9500)",
                               "border":"none","color":"#fff","padding":"0.4rem 0.7rem",
                               "borderRadius":"999px","cursor":"pointer","fontWeight":700}),
        ], style={"display":"flex","alignItems":"center"}),
    ], style={"display":"flex","justifyContent":"space-between","alignItems":"center",
              "padding":"0.75rem 1.5rem","borderBottom":"1px solid #4c4878",
              "background":"#13111f","color":"#f0ecff"}),

    dcc.Tabs(id="tabs", value="rank",
             parent_className="custom-tabs-container", className="custom-tabs",
             children=[
                 dcc.Tab(label="Rankings",       value="rank",   className="custom-tab", selected_className="custom-tab--selected"),
                 dcc.Tab(label="All Models",     value="models", className="custom-tab", selected_className="custom-tab--selected"),
                 dcc.Tab(label="Visualizations", value="viz",    className="custom-tab", selected_className="custom-tab--selected"),
                 dcc.Tab(label="Report",         value="report", className="custom-tab", selected_className="custom-tab--selected"),
             ]),
    html.Div(id="page-content", className="page"),
], className="app-shell")

@app.callback(Output("theme-store","data"), Input("theme-btn","n_clicks"),
              State("theme-store","data"), prevent_initial_call=True)
def toggle_theme(_, cur): return "light" if cur == "dark" else "dark"

@app.callback(Output("theme-btn","children"), Input("theme-store","data"))
def theme_icon(mode): return "☾" if mode == "light" else "☀"

@app.callback(Output("gpal-store","data"), Input("gpal-dropdown","value"), prevent_initial_call=True)
def set_gpal(v): return v

@app.callback(Output("page-content","children"),
              Input("tabs","value"), Input("theme-store","data"), Input("gpal-store","data"))
def render_page(tab, mode, gpal_name):
    c = theme(mode)
    shell = {"background":c["bg"],"color":c["text"],"minHeight":"100vh",
             "padding":"2rem","fontFamily":"Space Grotesk,sans-serif"}

    if tab == "rank":
        hero = html.Div([
            html.Div("Dual-Task Pipeline · Delay Analysis to Diagnosis Prediction",
                     style={"display":"inline-block","padding":"0.2rem 0.8rem","borderRadius":"999px",
                            "border":f"1px solid {c['accent']}","color":c["accent"],
                            "fontSize":"0.78rem","marginBottom":"0.6rem"}),
            html.H1(["Patient Journey ", html.Span("Bottleneck", style={"color":c["accent"]}), " Report"]),
            html.P("All 7 Spark MLlib models use a two-stage pipeline.",
                   style={"color":c["muted"],"maxWidth":"54rem"}),
            html.Div([stat_card(v,l,mode) for v,l in DATA["hero_stats"]],
                     style={"display":"flex","flexWrap":"wrap","gap":"0.8rem","marginTop":"1rem"}),
        ], style={"background":c["surface"],"border":f"1px solid {c['border']}",
                  "borderRadius":"18px","padding":"1.8rem","marginBottom":"1.5rem",
                  "boxShadow":"0 18px 40px rgba(0,0,0,0.55)"})

        rows = [html.Tr([
            html.Td(m["rank"]), html.Td(m["pair"]), html.Td(m["family"]),
            html.Td(star_rating(m["stars"])),
            html.Td(f"{m['r2']:.4f}"), html.Td(f"{m['rmse']:.2f}"),
            html.Td(f"{m['mae']:.2f}"), html.Td(f"{m['acc']}%"), html.Td(f"{m['f1']:.2f}"),
            html.Td(m["delay"], style={"fontSize":"0.78rem","maxWidth":"220px"}),
            html.Td(m["diag"],  style={"fontSize":"0.78rem","maxWidth":"200px"}),
        ]) for m in DATA["models"]]

        tbl = html.Div(html.Table([
            html.Thead(html.Tr([html.Th(h) for h in
                ["#","Model Pair","Family","Rating","LoS R2","RMSE","MAE",
                 "Diag Acc","Diag F1","Delay Finding","Diagnosis Note"]])),
            html.Tbody(rows)]),
            style={"overflowX":"auto","border":f"1px solid {c['border']}",
                   "borderRadius":"14px","background":c["surface"],"padding":"0.4rem 0.6rem"})
        return html.Div([hero, html.H2("Dual-Task Model Rankings"), tbl], style=shell)

    if tab == "models":
        p = list(GRAPH_PALETTES.values())[0]
        cards = [html.Div([
            html.Div([html.Span(f"{m['rank']}. {m['pair']}",
                                style={"fontWeight":600,"color":p[1%len(p)]}),
                      html.Span(f" · {m['summary']}", style={"color":c["muted"]})]),
            html.Div([html.Span("R2: "), html.Span(f"{m['r2']:.4f}   "),
                      html.Span("RMSE: "), html.Span(f"{m['rmse']:.2f}d   "),
                      html.Span("MAE: "), html.Span(f"{m['mae']:.2f}d   "),
                      html.Span("Acc: "), html.Span(f"{m['acc']}%   "),
                      html.Span("F1: "), html.Span(f"{m['f1']:.2f}")],
                     style={"marginTop":"0.4rem","fontFamily":"monospace","fontSize":"0.88rem"}),
            html.Div([html.Span("Delay: ", style={"fontWeight":600}), html.Span(m["delay"])],
                     style={"marginTop":"0.4rem","color":c["muted"],"fontSize":"0.85rem"}),
            html.Div([html.Span("Diagnosis: ", style={"fontWeight":600}),
                      html.Span(m["diag"], style={"color":p[0]})],
                     style={"marginTop":"0.2rem","fontSize":"0.85rem"}),
        ], style={"background":c["surface"],"border":f"1px solid {c['border']}",
                  "borderRadius":"14px","padding":"1rem 1.2rem","marginBottom":"0.9rem",
                  "boxShadow":"0 10px 28px rgba(0,0,0,0.45)"})
        for m in DATA["models"]]
        return html.Div([html.H2("All 7 Models"), *cards], style=shell)

    if tab == "viz":
        fig0 = build_figure("r2comp", "scatter", gpal_name, mode)
        return html.Div([
            html.H2("Result Visualizations"),
            html.P("Select chart group, graph style, and colour palette.",
                   style={"color":c["muted"],"maxWidth":"52rem","marginBottom":"1rem"}),
            html.Div([
                dcc.Dropdown(id="chart-type", clearable=False, className="dark-dd",
                    value="r2comp",
                    style={"backgroundColor":c["surface"],"color":c["text"]},
                    options=[
                        {"label":"LoS R2 vs Diagnosis Accuracy","value":"r2comp"},
                        {"label":"Delay Error RMSE vs MAE",     "value":"errcomp"},
                        {"label":"Before vs After Spark Uplift","value":"sparkrole"},
                    ]),
                dcc.Dropdown(id="graph-style", clearable=False, className="dark-dd",
                    value="scatter",
                    style={"backgroundColor":c["surface"],"color":c["text"]},
                    options=[
                        {"label":"Scatter Bubble","value":"scatter"},
                        {"label":"Radar Chart",   "value":"radar"},
                        {"label":"Grouped Bar",   "value":"grouped_bar"},
                        {"label":"Lollipop",      "value":"lollipop"},
                        {"label":"Waterfall",     "value":"waterfall"},
                        {"label":"Area Line",     "value":"area"},
                    ]),
                dcc.Dropdown(id="gpal-dropdown", clearable=False, className="dark-dd",
                    value=gpal_name,
                    style={"backgroundColor":c["surface"],"color":c["text"]},
                    options=[{"label":k,"value":k} for k in GRAPH_PALETTES]),
            ], style={"display":"grid","gridTemplateColumns":"1fr 1fr 1fr",
                      "gap":"1rem","marginBottom":"1.2rem"}),
            dcc.Graph(id="main-chart", figure=fig0,
                      style={"borderRadius":"14px","overflow":"hidden"}),
        ], style=shell)

    best      = min(DATA["models"], key=lambda m: m["rank"])
    best_r2   = max(m["r2"]   for m in DATA["models"])
    best_acc  = max(m["acc"]  for m in DATA["models"])
    best_rmse = min(m["rmse"] for m in DATA["models"])
    best_f1   = max(m["f1"]   for m in DATA["models"])
    figA, figB, figC = build_report_charts(mode)

    def kpi(label, value, last=False):
        return html.Div([
            html.Div(label, style={"fontSize":"0.72rem","color":c["muted"]}),
            html.Div(value, style={"fontSize":"1.45rem","fontWeight":700,"color":c["accent"]}),
        ], style={"flex":1,"padding":"0.8rem 1rem",
                  "borderRight":"none" if last else f"1px solid {c['border']}"})

    top_bar = html.Div([
        kpi("Total Patients","55,500"), kpi("Bottleneck Hospitals","22"),
        kpi("Best Diag Acc",f"{best_acc}%"), kpi("Best LoS R2",f"{best_r2:.3f}"),
        kpi("Best RMSE",f"{best_rmse:.2f}d"), kpi("Best F1",f"{best_f1:.2f}",last=True),
    ], style={"display":"flex","background":c["surface"],
              "border":f"1px solid {c['border']}","borderRadius":"16px","marginBottom":"1.5rem"})

    rec = html.Div([
        html.H3("Recommended Production Model", style={"marginBottom":"0.4rem"}),
        html.Div([
            html.Span(f"{best['short']} ({best['pair']})",
                      style={"color":c["accent"],"fontWeight":700,"fontSize":"1.05rem"}),
            html.Span(f"  R2 {best['r2']:.3f}  RMSE {best['rmse']:.2f}d  "
                      f"MAE {best['mae']:.2f}d  Acc {best['acc']}%  F1 {best['f1']:.2f}",
                      style={"color":c["muted"],"fontSize":"0.9rem"}),
        ]),
    ], style={"background":c["surface"],"border":f"1px solid {c['border']}",
              "borderRadius":"14px","padding":"1.1rem 1.4rem","marginBottom":"1.4rem"})

    def cbox(fig):
        return html.Div(dcc.Graph(figure=fig, style={"height":"320px"}),
            style={"flex":1,"background":c["surface"],"border":f"1px solid {c['border']}",
                   "borderRadius":"14px","overflow":"hidden","margin":"0 0.4rem"})

    return html.Div([
        html.H2("Multi-Hospital Bottleneck Report"),
        top_bar, rec,
        html.Div([cbox(figA), cbox(figB), cbox(figC)],
                 style={"display":"flex","margin":"0 -0.4rem"}),
    ], style=shell)

@app.callback(Output("main-chart","figure"),
              Input("chart-type","value"), Input("graph-style","value"),
              Input("theme-store","data"), Input("gpal-store","data"),
              prevent_initial_call=True)
def update_chart(ct, gs, mode, gpal_name):
    vm = {
        "r2comp":    {"scatter":"scatter","radar":"radar","grouped_bar":"grouped_bar",
                      "lollipop":"scatter","waterfall":"radar","area":"grouped_bar"},
        "errcomp":   {"scatter":"scatter","radar":"scatter","grouped_bar":"grouped_bar",
                      "lollipop":"grouped_bar","waterfall":"scatter","area":"area"},
        "sparkrole": {"scatter":"lollipop","radar":"lollipop","grouped_bar":"grouped_bar",
                      "lollipop":"lollipop","waterfall":"waterfall","area":"grouped_bar"},
    }
    return build_figure(ct, vm.get(ct,{}).get(gs, gs), gpal_name, mode)

PORT = 8461
def open_browser():
    if not os.environ.get("WERKZEUG_RUN_MAIN"):
        webbrowser.open_new(f"http://127.0.0.1:{PORT}/")

if __name__ == "__main__":
    Timer(1, open_browser).start()
    app.run(debug=False, port=PORT)